# FriskvårdsCentrum Nordic – ETL Pipeline
Denna notebook implementerar en komplett ETL‑pipeline med datatvätt, normalisering, sentimentanalys (BERT) och KPI‑visualisering.

## Importera bibliotek

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
from transformers import pipeline


## Ladda data

In [ ]:
df = pd.read_csv("friskvard_data.csv")
df_val = pd.read_csv("friskvard_validation.csv")

print("Rows main:", len(df))
print("Rows validation:", len(df_val))
df.head()


## Ta bort dubbletter

In [ ]:
df = df.drop_duplicates()


## Status normalisering

In [ ]:
status_map = {
    "genomförd":"completed",
    "klar":"completed",
    "done":"completed",
    "deltog":"completed",

    "avbokad":"cancelled",
    "avbokat":"cancelled",
    "cancelled":"cancelled",
    "canceled":"cancelled",
    "struken":"cancelled",

    "no show":"no_show",
    "noshow":"no_show",
    "no-show":"no_show",
    "uteblev":"no_show",
    "ej närvarande":"no_show",
    "missad":"no_show"
}

df["status"] = df["status"].str.lower().map(status_map)


## Normalisering av passnamn

In [ ]:
pass_map = {
"spinning":"spinning",
"spin":"spinning",
"indoor cycling":"spinning",
"cykel":"spinning",

"boxing":"boxing",
"boxning":"boxing",
"fightpass":"boxing",

"yoga":"yoga",
"hatha yoga":"yoga",
"vinyasa":"yoga",

"hiit":"hiit",
"h.i.i.t":"hiit",
"intervall":"hiit",
"högintensiv":"hiit",

"styrka":"strength",
"styrkepass":"strength",
"styrketräning":"strength",
"strength":"strength",

"dance":"dance",
"dans":"dance",
"zumba":"dance"
}

df["passnamn"] = df["passnamn"].str.lower().replace(pass_map)


## Normalisering av medlemstyp

In [ ]:
member_map = {
"bas":"basic",
"basic":"basic",
"grund":"basic",

"premium":"premium",
"plus":"premium",
"gold":"premium",

"student":"student",
"studerande":"student"
}

df["medlemstyp"] = df["medlemstyp"].str.lower().replace(member_map)


## Datumhantering

In [ ]:
df["passdatum"] = pd.to_datetime(df["passdatum"], errors="coerce", dayfirst=True)
df["bokningsdatum"] = pd.to_datetime(df["bokningsdatum"], errors="coerce", dayfirst=True)


## Logiska kontroller

In [ ]:
df = df[df["bokningsdatum"] <= df["passdatum"]]

df.loc[df["månadskostnad"] <= 0, "månadskostnad"] = None


## Feature Engineering

In [ ]:
df["weekday"] = df["passdatum"].dt.weekday
df["month"] = df["passdatum"].dt.month
df["age"] = 2024 - df["födelseår"]


## Sentimentanalys (BERT)

In [ ]:
sentiment_model = pipeline("sentiment-analysis")

def analyze_sentiment(text):
    if pd.isna(text):
        return None
    result = sentiment_model(str(text)[:512])[0]
    return result["label"].lower()

df["sentiment"] = df["feedback_text"].apply(analyze_sentiment)


## KPI 1 – Avbokningsfrekvens per månad

In [ ]:
cancel_rate = df.groupby("month")["status"].apply(lambda x: (x=="cancelled").mean())

cancel_rate.plot(kind="bar")
plt.title("Avbokningsfrekvens per månad")
plt.show()


## KPI 2 – Bokningar per veckodag

In [ ]:
bookings = df.groupby("weekday").size()

bookings.plot(kind="bar")
plt.title("Bokningar per veckodag")
plt.show()


## KPI 3 – Sentimentfördelning

In [ ]:
df["sentiment"].value_counts().plot(kind="bar")
plt.title("Sentiment i feedback")
plt.show()


## Spara till SQLite

In [ ]:
conn = sqlite3.connect("friskvard.db")

df.to_sql("bookings_clean", conn, if_exists="replace", index=False)
df_val.to_sql("validation_data", conn, if_exists="replace", index=False)

conn.close()
